# EoH HPC Compare Demo (Multi-Seed)

Run baseline vs routed with identical settings across multiple seeds, then generate aggregate plots.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

NB_DIR = Path.cwd().resolve()
PROJECT_ROOT = NB_DIR.parent if (NB_DIR / 'run_compare_baseline_routed.py').exists() else NB_DIR
COMPARE_ROOT = PROJECT_ROOT / 'compare_runs'

print('NOTEBOOK_DIR:', NB_DIR)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('COMPARE_ROOT:', COMPARE_ROOT)


## HPC + Experiment Settings

- Default HPC endpoint/key/model are set for ENSIA.
- `EOH_COMPARE_SEEDS` controls multi-seed runs (`comma,separated`).
- For faster debug, reduce `EOH_N_GENERATIONS` and/or use fewer seeds.

In [ ]:
# ENSIA HPC defaults
os.environ['ENSIA_VLLM_BASE'] = 'http://vllm-nodeport.vllm-ns.svc.cluster.local:8000/v1'
os.environ['ENSIA_VLLM_API_KEY'] = 'my-key-ensia-2022-1030'
os.environ['ENSIA_VLLM_MODEL'] = 'QuantTrio/Qwen3-VL-235B-A22B-Instruct-AWQ'

# Meeting demo defaults
os.environ['EOH_POP_SIZE'] = '8'
os.environ['EOH_N_GENERATIONS'] = '10'
os.environ['EOH_EVAL_INSTANCES_PER_GEN'] = '256'
os.environ['EOH_HOLDOUT_INSTANCES'] = '64'
os.environ['EOH_HOLDOUT_EVAL_INTERVAL'] = '1'
os.environ['EOH_ROUTE_IMPROVEMENT_EPS'] = '1e-12'
os.environ['EOH_ROUTE_WARMUP_GENS'] = '2'
os.environ['EOH_ROUTE_E1_COOLDOWN'] = '3'
os.environ['EOH_ROUTE_E2_RECENT_K'] = '3'
os.environ['EOH_ROUTE_USE_PROBABILISTIC'] = '1'
os.environ['EOH_N_PROC'] = '1'
os.environ['EOH_DISABLE_NUMBA'] = '1'
os.environ['EOH_LOG_LLM_IO'] = '1'
os.environ['EOH_COMPARE_OUT'] = str(COMPARE_ROOT)
os.environ['EOH_COMPARE_SEEDS'] = '2024,2025,2026'

print({k: os.environ.get(k) for k in [
    'ENSIA_VLLM_BASE',
    'ENSIA_VLLM_API_KEY',
    'ENSIA_VLLM_MODEL',
    'EOH_POP_SIZE',
    'EOH_N_GENERATIONS',
    'EOH_EVAL_INSTANCES_PER_GEN',
    'EOH_HOLDOUT_INSTANCES',
    'EOH_HOLDOUT_EVAL_INTERVAL',
    'EOH_ROUTE_IMPROVEMENT_EPS',
    'EOH_ROUTE_WARMUP_GENS',
    'EOH_ROUTE_E1_COOLDOWN',
    'EOH_ROUTE_E2_RECENT_K',
    'EOH_ROUTE_USE_PROBABILISTIC',
    'EOH_N_PROC',
    'EOH_DISABLE_NUMBA',
    'EOH_LOG_LLM_IO',
    'EOH_COMPARE_OUT',
    'EOH_COMPARE_SEEDS'
]})


In [ ]:
# QUICK TEST: routed-only (3 seeds, 10 generations) without re-running baseline
# Run this cell instead of the full compare cell when iterating fast.
import random
import numpy as np
from hpc_llm_setup import config_from_env, ensure_eoh_src_on_path, start_hpc_bridge, stop_hpc_bridge, test_bridge
from eoh import eoh
from eoh.utils.getParas import Paras

# quick overrides
os.environ['EOH_POP_SIZE'] = '8'
os.environ['EOH_N_GENERATIONS'] = '10'
os.environ['EOH_EVAL_INSTANCES_PER_GEN'] = '256'
os.environ['EOH_HOLDOUT_INSTANCES'] = '64'
os.environ['EOH_HOLDOUT_EVAL_INTERVAL'] = '1'
os.environ['EOH_N_PROC'] = '1'
os.environ['EOH_COMPARE_SEEDS'] = '2024,2025,2026'

_project_root, _ = ensure_eoh_src_on_path()
seeds = [int(s.strip()) for s in os.environ.get('EOH_COMPARE_SEEDS', '2024,2025,2026').split(',') if s.strip()]
print('routed quick seeds:', seeds)
cfg = config_from_env()
server = None
try:
    server, _thread, bridge_url, model_id = start_hpc_bridge(cfg)
    status, payload = test_bridge(bridge_url)
    print('bridge status:', status, payload)

    for seed in seeds:
        random.seed(seed)
        np.random.seed(seed)
        out_path = str(COMPARE_ROOT / f'seed_{seed}' / 'routed')
        paras = Paras()
        paras.set_paras(
            method='eoh',
            problem=os.environ.get('EOH_PROBLEM', 'bp_online'),
            llm_use_local=True,
            llm_local_url=bridge_url,
            llm_model=model_id,
            ec_pop_size=int(os.environ.get('EOH_POP_SIZE', '8')),
            ec_n_pop=int(os.environ.get('EOH_N_GENERATIONS', '10')),
            exp_n_proc=int(os.environ.get('EOH_N_PROC', '1')),
            exp_output_path=out_path,
            exp_debug_mode=False,
            eval_instances_per_gen=int(os.environ.get('EOH_EVAL_INSTANCES_PER_GEN', '256')),
            holdout_instances=int(os.environ.get('EOH_HOLDOUT_INSTANCES', '64')),
            holdout_eval_interval=int(os.environ.get('EOH_HOLDOUT_EVAL_INTERVAL', '1')),
            route_improvement_epsilon=float(os.environ.get('EOH_ROUTE_IMPROVEMENT_EPS', '1e-12')),
            route_warmup_gens=int(os.environ.get('EOH_ROUTE_WARMUP_GENS', '2')),
            route_e1_cooldown=int(os.environ.get('EOH_ROUTE_E1_COOLDOWN', '3')),
            route_e2_recent_k=int(os.environ.get('EOH_ROUTE_E2_RECENT_K', '3')),
            route_use_probabilistic=(os.environ.get('EOH_ROUTE_USE_PROBABILISTIC', '1') == '1'),
            eoh_mode='routed',
            log_full_population=False,
        )
        if os.environ.get('EOH_DISABLE_NUMBA', '1') == '1':
            paras.eva_numba_decorator = False

        print(f'running routed quick seed={seed} ->', out_path)
        runner = eoh.EVOL(paras)
        runner.run()
        print('run_log:', Path(out_path) / 'results' / 'run_log.jsonl')
        print('operator_log:', Path(out_path) / 'results' / 'operator_events.jsonl')
finally:
    stop_hpc_bridge(server)
    print('Bridge stopped')


In [ ]:
# QUICK PLOT: compare new routed quick run vs existing baseline from last run
seeds = [int(s.strip()) for s in os.environ.get('EOH_COMPARE_SEEDS', '2024,2025,2026').split(',') if s.strip()]
quick_logs = []
for seed in seeds:
    baseline_log = COMPARE_ROOT / f'seed_{seed}' / 'baseline' / 'results' / 'run_log.jsonl'
    routed_log = COMPARE_ROOT / f'seed_{seed}' / 'routed' / 'results' / 'run_log.jsonl'
    if baseline_log.exists():
        quick_logs.append(str(baseline_log))
    else:
        print('missing baseline log:', baseline_log)
    if routed_log.exists():
        quick_logs.append(str(routed_log))
    else:
        print('missing routed log:', routed_log)

if len(quick_logs) == 0:
    raise RuntimeError('No logs found for quick comparison')

quick_out = COMPARE_ROOT / 'quick_vs_last_baseline_plots'
cmd = [
    sys.executable,
    '-u',
    str(PROJECT_ROOT / 'notebooks' / 'plot_run_log.py'),
    '--logs',
] + quick_logs + [
    '--outdir',
    str(quick_out),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

from IPython.display import Image, display
for name in [
    'fitness_vs_gen.png',
    'operator_over_time.png',
    'invalid_and_diversity.png',
    'operator_effect_by_mode.png',
    'diagnosis_effect_routed.png',
]:
    p = quick_out / name
    print(name, 'exists=', p.exists())
    if p.exists():
        display(Image(filename=str(p)))


In [ ]:
# Run baseline+routed for each seed listed in EOH_COMPARE_SEEDS
status_log = COMPARE_ROOT / 'runner_status.jsonl'
print('Runner status log:', status_log)
cmd = [sys.executable, '-u', str(PROJECT_ROOT / 'notebooks' / 'run_compare_baseline_routed.py')]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)


In [ ]:
# Build aggregate plots from compare_runs (auto-discovers seed_* folders)
cmd = [
    sys.executable,
    '-u',
    str(PROJECT_ROOT / 'notebooks' / 'plot_run_log.py'),
    '--root',
    str(COMPARE_ROOT),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)


In [ ]:
from IPython.display import Image, display

plots_dir = COMPARE_ROOT / 'plots'
plot_files = [
    'fitness_vs_gen.png',
    'operator_over_time.png',
    'invalid_and_diversity.png',
    'operator_effect_by_mode.png',
    'diagnosis_effect_routed.png',
]

print('plots_dir:', plots_dir)
for name in plot_files:
    p = plots_dir / name
    print(name, 'exists=', p.exists())
    if p.exists():
        display(Image(filename=str(p)))


In [ ]:
# Quick tail of runner status for troubleshooting
status_log = COMPARE_ROOT / 'runner_status.jsonl'
if status_log.exists():
    lines = status_log.read_text(encoding='utf-8').splitlines()
    print('\n'.join(lines[-20:]))
else:
    print('No runner_status.jsonl yet')
